# Chapter 4: Tensors

**Book:** *Linear Algebra with Applications in Machine Learning: From Intuitive Understanding to Python Coding*

---

Tensors generalize scalars, vectors, and matrices to arbitrary dimensions. They are the natural data structure for images, videos, convolutional neural network weights, and any application involving multi-way data. This notebook covers:

1. **Definition and Basics** — order, shape, indexing, Einstein summation
2. **Tensor Operations** — addition, scalar multiplication, tensor product, contraction, mode-$n$ product, unfolding, norms, transformations
3. **Tensor Decompositions** — CP, Tucker, HOSVD, Tensor-Train
4. **Properties** — symmetry, orthogonality, rank
5. **Tensors in ML** — images, videos, CNN weights, EEG, networks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorlya as tl
from tensorly.decomposition import parafac, tucker

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
np.set_printoptions(precision=4, suppress=True)

print(f"NumPy version:   {np.__version__}")
print(f"TensorLy version: {tl.__version__}")
print("All imports successful.")

ModuleNotFoundError: No module named 'tensorly'

## 4.1 Tensors: Definition and Basics

A **tensor** is a multidimensional array characterized by its **order** (number of dimensions) and **shape** (size along each dimension):

| Object | Order | Example Shape | Indices |
|--------|-------|---------------|----------|
| Scalar | 0 | () | $a$ |
| Vector | 1 | $(n,)$ | $v_i$ |
| Matrix | 2 | $(m, n)$ | $A_{ij}$ |
| 3rd-order tensor | 3 | $(m, n, p)$ | $T_{ijk}$ |
| 4th-order tensor | 4 | $(m, n, p, q)$ | $S_{ijkl}$ |

In [ ]:
# Tensors of different orders
scalar = np.array(5)                                      # order 0
vector = np.array([1, 2, 3])                              # order 1
matrix = np.array([[1, 2], [3, 4]])                       # order 2
tensor3 = np.array([[[1,2],[3,4]], [[5,6],[7,8]]])        # order 3
tensor4 = np.random.randint(0, 10, size=(2, 2, 2, 2))    # order 4

objects = [('Scalar', scalar), ('Vector', vector), ('Matrix', matrix),
           ('3rd-order tensor', tensor3), ('4th-order tensor', tensor4)]

for name, obj in objects:
    print(f"{name:20s}  order={obj.ndim}  shape={obj.shape}")

### 3rd-Order Tensor: Indexing and Structure

A 3rd-order tensor $\mathbf{T} \in \mathbb{R}^{2 \times 2 \times 2}$ can be visualized as two $2 \times 2$ matrices stacked along the first axis. Each element is accessed by three indices: $T_{ijk}$ (layer, row, column).

In [ ]:
T = np.array([[[1, 2], [3, 4]],
              [[5, 6], [7, 8]]])

print(f"Shape: {T.shape}")
print(f"\nLayer 0 (T[0,:,:]):\n{T[0]}")
print(f"\nLayer 1 (T[1,:,:]):\n{T[1]}")

print(f"\nElement access:")
print(f"  T[0,0,0] = {T[0,0,0]}  (layer 0, row 0, col 0)")
print(f"  T[0,0,1] = {T[0,0,1]}  (layer 0, row 0, col 1)")
print(f"  T[1,1,0] = {T[1,1,0]}  (layer 1, row 1, col 0)")
print(f"  T[1,1,1] = {T[1,1,1]}  (layer 1, row 1, col 1)")

### Visualizing a 3rd-Order Tensor

We can display the two "slices" (layers) of the tensor side by side as heatmaps.

In [ ]:
T = np.array([[[1, 2], [3, 4]],
              [[5, 6], [7, 8]]])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for k in range(2):
    ax = axes[k]
    im = ax.imshow(T[k], cmap='YlOrRd', vmin=0, vmax=9)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(T[k, i, j]), ha='center', va='center',
                    fontsize=18, fontweight='bold')
    ax.set_title(f'Layer {k}  (T[{k},:,:])', fontsize=13)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlabel('Column (k)')
    ax.set_ylabel('Row (j)')

plt.suptitle(r'3rd-Order Tensor $\mathbf{T} \in \mathbb{R}^{2 \times 2 \times 2}$',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Einstein Summation Convention

The Einstein convention implies summation over repeated indices. NumPy's `np.einsum` implements this directly.

**Matrix multiplication:** $C_{ij} = A_{ik} B_{kj}$ (sum over $k$)

**Tensor-vector contraction:** $R_{ij} = T_{ijk} v_k$ (sum over $k$)

In [ ]:
# Matrix multiplication via einsum
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

C_standard = A @ B
C_einsum = np.einsum('ik,kj->ij', A, B)

print(f"A @ B =\n{C_standard}")
print(f"einsum('ik,kj->ij') =\n{C_einsum}")
print(f"Match: {np.allclose(C_standard, C_einsum)}")

In [ ]:
# Quadruple contraction: S_{ijkl} u_i v_j w_k x_l  (all-ones S)
S = np.ones((2, 2, 2, 2))
u = np.array([1, 2])
v = np.array([0, 1])
w = np.array([1, 1])
x = np.array([2, 1])

result = np.einsum('ijkl,i,j,k,l->', S, u, v, w, x)

# Verify: since S is all ones, result = sum(u)*sum(v)*sum(w)*sum(x)
expected = u.sum() * v.sum() * w.sum() * x.sum()
print(f"Quadruple contraction: {result}")
print(f"Expected (product of sums): {expected}")

## 4.2 Tensor Operations

### 4.2.1 Addition, Subtraction, and Scalar Multiplication

These are element-wise operations, exactly as with vectors and matrices, but extended to any number of dimensions. Both tensors must have the same shape.

In [ ]:
A = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
B = np.array([[[0, 1], [1, 0]], [[1, 0], [0, 1]]])

print(f"A + B =\n{A + B}\n")
print(f"A - B =\n{A - B}\n")
print(f"2 * A =\n{2 * A}")

### Hadamard (Element-wise) Product for Tensors

Just as with matrices, the Hadamard product multiplies corresponding elements.

In [ ]:
hadamard = A * B
print(f"A (Hadamard) B =\n{hadamard}")

### 4.2.2 Tensor Product (Kronecker Product)

The tensor product $\mathbf{A} \otimes \mathbf{B}$ combines two tensors into a higher-dimensional structure. For matrices, this is the Kronecker product.

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[0, 1], [1, 0]])

kron = np.kron(A, B)
print(f"A =\n{A}\n")
print(f"B =\n{B}\n")
print(f"A (kron) B =\n{kron}")

In [ ]:
# Visualize the Kronecker product structure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, mat, title in zip(axes, [A, B, kron], ['A (2x2)', 'B (2x2)', r'$A \otimes B$ (4x4)']):
    im = ax.imshow(mat, cmap='coolwarm', vmin=-1, vmax=5)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, str(mat[i, j]), ha='center', va='center',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontsize=12)
    ax.set_xticks(range(mat.shape[1]))
    ax.set_yticks(range(mat.shape[0]))

plt.suptitle('Tensor (Kronecker) Product', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Outer Product for Vectors

The outer product of two vectors produces a rank-1 matrix. The outer product of three vectors produces a rank-1 3rd-order tensor.

In [ ]:
u = np.array([1, 2])
v = np.array([3, 4])
w = np.array([5, 6])

# Rank-1 matrix
outer_2d = np.outer(u, v)
print(f"u (x) v (rank-1 matrix):\n{outer_2d}\n")

# Rank-1 3rd-order tensor via einsum
outer_3d = np.einsum('i,j,k->ijk', u, v, w)
print(f"u (x) v (x) w (rank-1 tensor, shape {outer_3d.shape}):\n{outer_3d}")

### 4.2.3 Tensor Contraction

Contraction reduces a tensor's order by summing over one or more indices. It generalizes the dot product and matrix multiplication.

For a 3rd-order tensor $T_{ijk}$ contracted with vector $v_k$:
$$R_{ij} = \sum_k T_{ijk} \, v_k$$

In [ ]:
# Tensor-vector contraction
T = np.ones((2, 2, 3))
v = np.array([1, 2, 3])

R = np.tensordot(T, v, axes=([2], [0]))
R_einsum = np.einsum('ijk,k->ij', T, v)

print(f"T shape: {T.shape}, v shape: {v.shape}")
print(f"Contraction result (tensordot):\n{R}")
print(f"Contraction result (einsum):\n{R_einsum}")
print(f"Match: {np.allclose(R, R_einsum)}")

In [ ]:
# Double contraction: S_{ijkl} A_{kl} -> R_{ij}
S = np.ones((2, 2, 2, 2))
A = np.array([[1, 2], [3, 4]])

R = np.tensordot(S, A, axes=([2, 3], [0, 1]))
print(f"Double contraction result:\n{R}")
print(f"Each entry = sum of all elements in A = {A.sum()}")

### 4.2.4 Mode-$n$ Product

The mode-$n$ product multiplies a tensor by a matrix along its $n$-th mode. If $\mathbf{T} \in \mathbb{R}^{I_1 \times I_2 \times I_3}$ and $\mathbf{A} \in \mathbb{R}^{J \times I_n}$, the result replaces the $n$-th dimension size $I_n$ with $J$.

In [ ]:
T = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
A = np.array([[1, 2], [3, 4]])

print(f"T shape: {T.shape}")
print(f"A shape: {A.shape}")

# Mode-0 product: A transforms the first mode
result_mode0 = tl.tenalg.mode_dot(T, A, mode=0)
print(f"\nMode-0 product shape: {result_mode0.shape}")
print(f"Result:\n{result_mode0}")

# Mode-1 product: A transforms the second mode
result_mode1 = tl.tenalg.mode_dot(T, A, mode=1)
print(f"\nMode-1 product shape: {result_mode1.shape}")
print(f"Result:\n{result_mode1}")

### 4.2.5 Tensor Unfolding (Matricization)

Unfolding rearranges a tensor into a matrix by "flattening" along one mode. The mode-$n$ unfolding $T_{(n)}$ has rows indexed by the $n$-th mode and columns indexed by all other modes. This is a key step in many decomposition algorithms.

In [ ]:
T = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

print(f"Original tensor T, shape {T.shape}:")
print(T)

for mode in range(3):
    unfolded = tl.unfold(T, mode=mode)
    print(f"\nMode-{mode} unfolding, shape {unfolded.shape}:")
    print(unfolded)

In [ ]:
# Visualize all three unfoldings
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for mode, ax in enumerate(axes):
    unf = tl.unfold(T, mode=mode)
    im = ax.imshow(unf, cmap='YlOrRd', vmin=0, vmax=9)
    for i in range(unf.shape[0]):
        for j in range(unf.shape[1]):
            ax.text(j, i, str(unf[i, j]), ha='center', va='center',
                    fontsize=13, fontweight='bold')
    ax.set_title(f'Mode-{mode} Unfolding  ({unf.shape[0]}x{unf.shape[1]})', fontsize=12)
    ax.set_xlabel('Combined indices')
    ax.set_ylabel(f'Mode-{mode} index')

plt.suptitle('Tensor Unfolding (Matricization)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Refolding

We can refold a matrix back into the original tensor shape, verifying that unfolding is a reversible rearrangement.

In [ ]:
for mode in range(3):
    unfolded = tl.unfold(T, mode=mode)
    refolded = tl.fold(unfolded, mode=mode, shape=T.shape)
    print(f"Mode-{mode}: unfold then refold matches original? {np.allclose(T, refolded)}")

### 4.2.6 Tensor Norms

The **Frobenius norm** treats the tensor as a flattened vector and computes its Euclidean norm:

$$\|\mathbf{T}\|_F = \sqrt{\sum_{i,j,k} T_{ijk}^2}$$

Other norms include the nuclear norm (sum of singular values of an unfolding) and the spectral norm (largest singular value).

In [ ]:
T = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

# Frobenius norm
frob = np.linalg.norm(T)
frob_manual = np.sqrt(np.sum(T**2))
print(f"Frobenius norm: {frob:.4f}")
print(f"Manual check:   {frob_manual:.4f}")

# L1 norm (sum of absolute values)
l1_norm = np.sum(np.abs(T))
print(f"L1 norm:        {l1_norm}")

# Nuclear and spectral norms via mode-0 unfolding
T_unf = tl.unfold(T, mode=0)
singular_values = np.linalg.svd(T_unf, compute_uv=False)
nuclear = np.sum(singular_values)
spectral = np.max(singular_values)

print(f"\nSingular values of mode-0 unfolding: {singular_values}")
print(f"Nuclear norm:   {nuclear:.4f}")
print(f"Spectral norm:  {spectral:.4f}")

### 4.2.7 Tensor Transformations

Under a change of basis described by an orthogonal matrix $Q$, a 2nd-order tensor transforms as $A' = Q^T A Q$. This preserves geometric properties (eigenvalues, trace, etc.).

In [ ]:
A = np.array([[1, 2], [2, 3]])

# 45-degree rotation
theta = np.pi / 4
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

A_prime = Q.T @ A @ Q

print(f"Original A:\n{A}\n")
print(f"Rotation Q (theta={np.degrees(theta):.0f} deg):\n{Q.round(4)}\n")
print(f"Transformed A' = Q^T A Q:\n{A_prime.round(4)}\n")

# Invariants preserved under orthogonal transformation
print(f"Trace(A)  = {np.trace(A):.4f},  Trace(A') = {np.trace(A_prime):.4f}")
print(f"Det(A)    = {np.linalg.det(A):.4f},  Det(A')   = {np.linalg.det(A_prime):.4f}")
print(f"Eigenvalues(A):  {np.sort(np.linalg.eigvalsh(A))}")
print(f"Eigenvalues(A'): {np.sort(np.linalg.eigvalsh(A_prime))}")

## 4.3 Tensor Decompositions

Decompositions factor a tensor into simpler components, enabling compression, denoising, and latent structure extraction.

### 4.3.1 CP (CANDECOMP/PARAFAC) Decomposition

CP approximates a tensor as a sum of $R$ rank-one tensors:

$$T_{ijk} \approx \sum_{r=1}^{R} a_r^{(1)}{}_i \;\; a_r^{(2)}{}_j \;\; a_r^{(3)}{}_k$$

Each rank-one component is the outer product of three vectors. The minimum $R$ for exact reconstruction is the **tensor rank**.

In [ ]:
np.random.seed(42)
T = np.random.rand(4, 3, 3)

# CP decomposition with rank 3
rank = 3
cp_result = parafac(T, rank=rank)
weights, factors = cp_result

print(f"Original tensor shape: {T.shape}")
print(f"CP rank: {rank}")
print(f"Factor matrix shapes: {[f.shape for f in factors]}")
print(f"Weights: {weights}")

# Reconstruct
T_reconstructed = tl.cp_to_tensor(cp_result)
error = np.linalg.norm(T - T_reconstructed) / np.linalg.norm(T)
print(f"\nRelative reconstruction error: {error:.6f}")

In [ ]:
# Visualize: original vs reconstructed (first two slices)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for k, label in enumerate(['Layer 0', 'Layer 1']):
    im1 = axes[k, 0].imshow(T[k], cmap='viridis', vmin=0, vmax=1)
    axes[k, 0].set_title(f'Original {label}', fontsize=12)
    for i in range(T.shape[1]):
        for j in range(T.shape[2]):
            axes[k, 0].text(j, i, f'{T[k,i,j]:.2f}', ha='center', va='center', fontsize=10)

    im2 = axes[k, 1].imshow(T_reconstructed[k], cmap='viridis', vmin=0, vmax=1)
    axes[k, 1].set_title(f'CP Reconstructed {label} (rank {rank})', fontsize=12)
    for i in range(T_reconstructed.shape[1]):
        for j in range(T_reconstructed.shape[2]):
            axes[k, 1].text(j, i, f'{T_reconstructed[k,i,j]:.2f}',
                           ha='center', va='center', fontsize=10)

plt.suptitle('CP Decomposition: Original vs. Reconstructed', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Effect of Rank on Reconstruction Quality

Lower rank means more compression but higher error. Let us sweep ranks and plot the trade-off.

In [ ]:
np.random.seed(0)
T_big = np.random.rand(8, 6, 5)

ranks = range(1, 6)
errors = []
for r in ranks:
    cp_r = parafac(T_big, rank=r)
    T_r = tl.cp_to_tensor(cp_r)
    rel_err = np.linalg.norm(T_big - T_r) / np.linalg.norm(T_big)
    errors.append(rel_err)
    print(f"Rank {r}: relative error = {rel_err:.6f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(ranks), errors, 'bo-', markersize=8, linewidth=2)
ax.set_xlabel('CP Rank')
ax.set_ylabel('Relative Reconstruction Error')
ax.set_title('CP Decomposition: Rank vs. Error Trade-off')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3.2 Tucker Decomposition

Tucker decomposition factors a tensor into a **core tensor** $\mathbf{G}$ and factor matrices along each mode:

$$T_{ijk} \approx \sum_{p,q,r} G_{pqr} \, A_{ip} \, B_{jq} \, C_{kr}$$

Unlike CP, the core tensor $\mathbf{G}$ can be dense and captures interactions between components.

In [ ]:
T = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]], dtype=float)

# Full-rank Tucker (no compression)
core, factors = tucker(T, rank=[2, 2, 2])
T_tucker = tl.tucker_to_tensor((core, factors))

print(f"Original shape: {T.shape}")
print(f"Core shape:     {core.shape}")
print(f"Factor shapes:  {[f.shape for f in factors]}")
print(f"\nReconstruction error: {np.linalg.norm(T - T_tucker):.8f}")
print(f"\nCore tensor:\n{core.round(4)}")

### 4.3.3 HOSVD (Higher-Order SVD)

HOSVD is a special case of Tucker decomposition where the factor matrices are obtained by applying SVD to each mode-$n$ unfolding. The factors are orthogonal.

In [ ]:
np.random.seed(7)
T = np.random.rand(3, 3, 3)

# HOSVD via Tucker with full rank
core, factors = tucker(T, rank=[3, 3, 3])
T_hosvd = tl.tucker_to_tensor((core, factors))

print(f"Reconstructed shape: {T_hosvd.shape}")
print(f"Reconstruction error: {np.linalg.norm(T - T_hosvd):.10f}")

# Check orthogonality of factor matrices
for i, f in enumerate(factors):
    orth_check = f.T @ f
    print(f"Factor {i}: orthogonal? {np.allclose(orth_check, np.eye(f.shape[1]), atol=1e-10)}")

### 4.3.4 Tensor-Train (TT) Decomposition

TT decomposition represents a tensor as a chain (train) of 3rd-order core tensors. For a tensor of order $N$:

$$T_{i_1 i_2 \cdots i_N} \approx \sum_{r_1,\dots,r_{N-1}} G^{(1)}_{1,i_1,r_1} G^{(2)}_{r_1,i_2,r_2} \cdots G^{(N)}_{r_{N-1},i_N,1}$$

TT decomposition scales linearly in the number of dimensions, making it efficient for very high-order tensors.

In [ ]:
from tensorly.decomposition import tensor_train

np.random.seed(42)
T = np.random.rand(2, 2, 2, 2)

# TT decomposition
tt_cores = tensor_train(T, rank=[1, 2, 2, 2, 1])

print(f"Original shape: {T.shape}")
print(f"Number of TT cores: {len(tt_cores)}")
for i, core in enumerate(tt_cores):
    print(f"  Core {i} shape: {core.shape}")

# Reconstruct
T_tt = tl.tt_to_tensor(tt_cores)
error = np.linalg.norm(T - T_tt) / np.linalg.norm(T)
print(f"\nRelative reconstruction error: {error:.10f}")

### Decomposition Comparison Summary

| Decomposition | Factors | Strengths | Use Cases |
|---------------|---------|-----------|----------|
| CP | Sum of rank-1 tensors | Uniqueness (under mild conditions), interpretable | Recommender systems, chemometrics |
| Tucker | Core + factor matrices | Flexible multilinear rank | Image compression, feature extraction |
| HOSVD | Orthogonal Tucker | Orthogonal factors, truncation-based | Initialization for Tucker, denoising |
| Tensor-Train | Chain of 3rd-order cores | Linear scaling in order | High-dimensional PDEs, quantum computing |

## 4.4 Properties of Tensors

### 4.4.1 Symmetric Tensors

A tensor is **symmetric** if its entries are invariant under any permutation of indices. For a 3rd-order tensor: $T_{ijk} = T_{jki} = T_{kij} = \cdots$

In [ ]:
# Construct a symmetric 3rd-order tensor
T_sym = np.array([[[1, 2], [2, 3]],
                   [[2, 3], [3, 4]]])

# Check all permutations
permutations = [
    ('ijk', T_sym),
    ('jki', T_sym.transpose(1, 2, 0)),
    ('kij', T_sym.transpose(2, 0, 1)),
    ('ikj', T_sym.transpose(0, 2, 1)),
    ('jik', T_sym.transpose(1, 0, 2)),
    ('kji', T_sym.transpose(2, 1, 0)),
]

print("Symmetry check (all permutations equal):")
all_equal = True
for perm_name, perm_tensor in permutations:
    match = np.allclose(T_sym, perm_tensor)
    all_equal = all_equal and match
    print(f"  T_{perm_name} == T_ijk? {match}")
print(f"Tensor is symmetric: {all_equal}")
print(f"\nIndependent entries: {4} out of {T_sym.size} total")

### 4.4.2 Orthogonal Tensors (2nd-order)

An orthogonal matrix $Q$ satisfies $Q^T Q = I$. It represents rotations or reflections that preserve lengths and angles.

In [ ]:
# 90-degree rotation matrix
Q = np.array([[0, -1],
              [1,  0]])

print(f"Q =\n{Q}\n")
print(f"Q^T Q =\n{Q.T @ Q}  (should be I)\n")

# Length preservation
v = np.array([3, 4])
Qv = Q @ v
print(f"v = {v},  ||v|| = {np.linalg.norm(v):.4f}")
print(f"Qv = {Qv}, ||Qv|| = {np.linalg.norm(Qv):.4f}")
print(f"Length preserved: {np.isclose(np.linalg.norm(v), np.linalg.norm(Qv))}")

### 4.4.3 Tensor Rank

The rank of a tensor is the minimum number of rank-one tensors needed for exact CP decomposition. Unlike matrix rank, tensor rank is NP-hard to compute exactly, so in practice we use iterative methods.

In [ ]:
# Construct a rank-2 tensor explicitly
u1, v1, w1 = np.array([1, 2]), np.array([3, 4]), np.array([5, 6])
u2, v2, w2 = np.array([1, 1]), np.array([1, 1]), np.array([1, 1])

T1 = np.einsum('i,j,k->ijk', u1, v1, w1)
T2 = np.einsum('i,j,k->ijk', u2, v2, w2)
T_rank2 = T1 + T2

print(f"Rank-1 component T1:\n{T1}\n")
print(f"Rank-1 component T2:\n{T2}\n")
print(f"T = T1 + T2 (rank 2):\n{T_rank2}\n")

# Try to recover with CP rank 2
cp_r2 = parafac(T_rank2.astype(float), rank=2)
T_recovered = tl.cp_to_tensor(cp_r2)
print(f"CP rank-2 reconstruction error: {np.linalg.norm(T_rank2 - T_recovered):.8f}")

## 4.5 Tensors in Machine Learning

### 4.5.1 Images as 3rd-Order Tensors

A color image is a 3rd-order tensor of shape (height, width, channels). Each channel is a 2D matrix of pixel intensities.

In [ ]:
# Create a small synthetic RGB image
image = np.zeros((4, 4, 3), dtype=np.uint8)
image[:2, :2, 0] = 255       # Red in top-left
image[:2, 2:, 1] = 255       # Green in top-right
image[2:, :2, 2] = 255       # Blue in bottom-left
image[2:, 2:] = [255, 255, 0]  # Yellow in bottom-right

print(f"Image tensor shape: {image.shape}  (height x width x RGB)")

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

axes[0].imshow(image)
axes[0].set_title('RGB Image', fontsize=12)

channel_names = ['Red', 'Green', 'Blue']
cmaps = ['Reds', 'Greens', 'Blues']
for c in range(3):
    axes[c+1].imshow(image[:, :, c], cmap=cmaps[c], vmin=0, vmax=255)
    axes[c+1].set_title(f'{channel_names[c]} Channel', fontsize=12)
    for i in range(4):
        for j in range(4):
            axes[c+1].text(j, i, str(image[i, j, c]), ha='center', va='center',
                          fontsize=9)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Color Image as a 3rd-Order Tensor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.5.2 CNN Weights as 4th-Order Tensors

In a convolutional neural network, each convolutional layer has weights shaped as (output_channels, input_channels, kernel_height, kernel_width). This 4th-order tensor encodes all learned spatial filters.

In [ ]:
np.random.seed(0)
# Simulated CNN layer: 8 output filters, 4 input channels, 3x3 kernel
cnn_weights = np.random.randn(8, 4, 3, 3)

print(f"CNN weight tensor shape: {cnn_weights.shape}")
print(f"  Output channels (filters): {cnn_weights.shape[0]}")
print(f"  Input channels:            {cnn_weights.shape[1]}")
print(f"  Kernel size:               {cnn_weights.shape[2]}x{cnn_weights.shape[3]}")
print(f"  Total parameters:          {cnn_weights.size}")

# Visualize the first output filter's kernels across input channels
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for c in range(4):
    im = axes[c].imshow(cnn_weights[0, c], cmap='RdBu_r', vmin=-2, vmax=2)
    axes[c].set_title(f'Filter 0, Input Ch {c}', fontsize=11)
    axes[c].set_xticks([])
    axes[c].set_yticks([])

plt.suptitle('CNN Weights: Filter 0 across Input Channels', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.5.3 Compressing CNN Weights with Tucker Decomposition

Tucker decomposition can compress CNN weight tensors, reducing memory and computation while approximately preserving the learned filters.

In [ ]:
# Compress the CNN weights with reduced Tucker rank
core, factors = tucker(cnn_weights, rank=[4, 2, 3, 3])
cnn_compressed = tl.tucker_to_tensor((core, factors))

orig_params = cnn_weights.size
compressed_params = core.size + sum(f.size for f in factors)

error = np.linalg.norm(cnn_weights - cnn_compressed) / np.linalg.norm(cnn_weights)

print(f"Original parameters:   {orig_params}")
print(f"Compressed parameters: {compressed_params}")
print(f"Compression ratio:     {orig_params / compressed_params:.2f}x")
print(f"Relative error:        {error:.4f}")

### 4.5.4 Other ML Tensor Applications

In [ ]:
# Video: 4th-order tensor (frames x height x width x channels)
video = np.random.randint(0, 256, size=(30, 64, 64, 3), dtype=np.uint8)
print(f"Video tensor:     {video.shape}  (30 frames, 64x64 RGB)")

# EEG data: 3rd-order tensor (channels x time x trials)
eeg = np.random.randn(32, 1000, 10)
print(f"EEG tensor:       {eeg.shape}  (32 channels, 1000 time points, 10 trials)")

# Feature map: 3rd-order tensor (height x width x features)
feature_map = np.random.rand(64, 64, 128)
print(f"Feature map:      {feature_map.shape}  (64x64 spatial, 128 feature channels)")

# MRI multi-modal: 3rd-order tensor (height x width x modalities)
mri = np.random.rand(256, 256, 3)
print(f"MRI tensor:       {mri.shape}  (256x256, 3 modalities)")

# Social network over time: 3rd-order tensor (nodes x nodes x time)
network = np.random.randint(0, 2, size=(50, 50, 10))
print(f"Network tensor:   {network.shape}  (50 nodes, 10 time steps)")

In [ ]:
# CP decomposition of EEG data to extract latent components
np.random.seed(42)
eeg_small = np.random.rand(32, 100, 10)  # smaller for speed

cp_eeg = parafac(eeg_small, rank=5)
weights_eeg, factors_eeg = cp_eeg

print(f"EEG CP decomposition (rank 5):")
print(f"  Channel factor shape:  {factors_eeg[0].shape}  (32 channels x 5 components)")
print(f"  Time factor shape:     {factors_eeg[1].shape}  (100 time points x 5 components)")
print(f"  Trial factor shape:    {factors_eeg[2].shape}  (10 trials x 5 components)")

# Plot the temporal patterns
fig, ax = plt.subplots(figsize=(10, 4))
for r in range(5):
    ax.plot(factors_eeg[1][:, r], label=f'Component {r+1}', alpha=0.8)
ax.set_xlabel('Time Point')
ax.set_ylabel('Loading')
ax.set_title('CP Decomposition of EEG: Temporal Components', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.6 Exercises

Selected exercises from the chapter. Try them below, then check the solutions.

**Exercise 1:** Compute $\mathbf{A} + \mathbf{B}$ for the 3rd-order tensors given in the textbook.

In [ ]:
# Exercise 1: Your code here


**Exercise 4:** Perform a tensor contraction between $\mathbf{T}$ (shape 2x2x3) and $\mathbf{v} = [1, 2, 3]$.

In [ ]:
# Exercise 4: Your code here


**Exercise 9:** Calculate the Frobenius norm of $\mathbf{T} = [[[1,2],[3,4]],[[5,6],[7,8]]]$.

In [ ]:
# Exercise 9: Your code here


**Exercise 10:** Transform $A = \begin{bmatrix} 1 & 2 \\ 2 & 3 \end{bmatrix}$ using a rotation matrix with $\theta = \pi/4$.

In [ ]:
# Exercise 10: Your code here


---

## Exercise Solutions

In [ ]:
# --- Solution: Exercise 1 ---
A = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
B = np.array([[[1, 0], [0, 1]], [[0, 1], [1, 0]]])
print(f"A + B =\n{A + B}")

In [ ]:
# --- Solution: Exercise 4 ---
T = np.array([[[1, 2, 3], [4, 5, 6]],
              [[7, 8, 9], [10, 11, 12]]])
v = np.array([1, 2, 3])

R = np.einsum('ijk,k->ij', T, v)
print(f"Tensor contraction T_ijk * v_k:\n{R}")
print(f"\nVerification: T[0,0,:] . v = {T[0,0,:] @ v}")

In [ ]:
# --- Solution: Exercise 9 ---
T = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
frob = np.linalg.norm(T)
print(f"Frobenius norm = sqrt({np.sum(T**2)}) = {frob:.4f}")

In [ ]:
# --- Solution: Exercise 10 ---
A = np.array([[1, 2], [2, 3]])
theta = np.pi / 4
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

A_prime = Q.T @ A @ Q
print(f"A' = Q^T A Q =\n{A_prime.round(4)}")
print(f"\nTrace preserved: {np.isclose(np.trace(A), np.trace(A_prime))}")
print(f"Det preserved:   {np.isclose(np.linalg.det(A), np.linalg.det(A_prime))}")